# Bronze Ingestion - Customers

## Objective

Ingest customer data from the Databricks sample dataset into the Bronze layer while preserving source data and adding technical metadata for traceability.

## Source

`samples.bakehouse.sales_customers`

## Target

`workspace.retail_bronze.customers`

## Engineering Concepts

- Source-to-Bronze ingestion
- Spark DataFrames
- Technical metadata
- Batch identifiers
- Managed Delta tables
- Basic ingestion validation

In [0]:
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime

source_table = "samples.bakehouse.sales_customers"
target_table = "workspace.retail_bronze.customers"

batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Batch ID: {batch_id}")
print(f"Source: {source_table}")
print(f"Target: {target_table}")

In [0]:
df_source = spark.table(source_table)

display(df_source.limit(10))

In [0]:
df_bronze = (
    df_source
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_batch_id", lit(batch_id))
    .withColumn("_source_system", lit("databricks_samples"))
    .withColumn("_source_table", lit(source_table))
)

In [0]:
source_count = df_source.count()

print(f"Source row count: {source_count}")

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)

In [0]:
df_target = spark.table(target_table)

target_count = df_target.count()

print(f"Source count: {source_count}")
print(f"Target count: {target_count}")

if source_count == target_count:
    print("PASS: Bronze ingestion row counts match.")
else:
    raise Exception(
        f"FAILED: Source count {source_count} does not match target count {target_count}"
    )

In [0]:
DESCRIBE HISTORY workspace.retail_bronze.customers;